In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

# 데이터 로드
X = np.load('../data/X_resampled.npy')
y = np.load('../data/y_resampled.npy')

# train/test split (논문이랑 똑같이 80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (16526, 17), Test: (4132, 17)


## 모델 학습

In [3]:
# 결과 저장할 딕셔너리
results = {}

# 1. SVM
print('SVM 학습 중...')
svm = SVC(kernel='rbf', C=50.12, gamma=0.120, probability=True, random_state=42)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_proba = svm.predict_proba(X_test)[:, 1]
results['SVM'] = {
    'report': classification_report(y_test, svm_pred, output_dict=True),
    'roc_auc': roc_auc_score(y_test, svm_proba)
}
print('SVM 완료!')

# 2. XGBoost
print('XGBoost 학습 중...')
xgb = XGBClassifier(max_depth=4, learning_rate=0.11, subsample=0.8, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
results['XGBoost'] = {
    'report': classification_report(y_test, xgb_pred, output_dict=True),
    'roc_auc': roc_auc_score(y_test, xgb_proba)
}
print('XGBoost 완료!')

# 3. CatBoost
print('CatBoost 학습 중...')
cat = CatBoostClassifier(iterations=245, learning_rate=0.09, max_depth=8, verbose=0, random_state=42)
cat.fit(X_train, y_train)
cat_pred = cat.predict(X_test)
cat_proba = cat.predict_proba(X_test)[:, 1]
results['CatBoost'] = {
    'report': classification_report(y_test, cat_pred, output_dict=True),
    'roc_auc': roc_auc_score(y_test, cat_proba)
}
print('CatBoost 완료!')

# 4. BPANN
print('BPANN 학습 중...')
bpann = MLPClassifier(hidden_layer_sizes=(100, 100, 100), max_iter=300, random_state=42)
bpann.fit(X_train, y_train)
bpann_pred = bpann.predict(X_test)
bpann_proba = bpann.predict_proba(X_test)[:, 1]
results['BPANN'] = {
    'report': classification_report(y_test, bpann_pred, output_dict=True),
    'roc_auc': roc_auc_score(y_test, bpann_proba)
}
print('BPANN 완료!')

print('\n모든 모델 학습 완료!')

SVM 학습 중...
SVM 완료!
XGBoost 학습 중...
XGBoost 완료!
CatBoost 학습 중...
CatBoost 완료!
BPANN 학습 중...
BPANN 완료!

모든 모델 학습 완료!


## 결과 출력


In [4]:
# 결과 테이블 출력
summary = []
for model_name, result in results.items():
    summary.append({
        'Model': model_name,
        'Precision': round(result['report']['weighted avg']['precision'], 3),
        'Recall': round(result['report']['weighted avg']['recall'], 3),
        'F1-Score': round(result['report']['weighted avg']['f1-score'], 3),
        'ROC AUC': round(result['roc_auc'], 3)
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

   Model  Precision  Recall  F1-Score  ROC AUC
     SVM      0.942   0.941     0.941    0.974
 XGBoost      0.939   0.939     0.939    0.986
CatBoost      0.947   0.947     0.947    0.990
   BPANN      0.938   0.936     0.936    0.969


모델 주요 특징 정리 및 성능 비교

| Model | 주요 특징 |
|---|---|
| SVM | 고차원 데이터에 강함, 소규모 데이터에 적합 |
| XGBoost | 대규모 데이터 처리, 빠른 속도 |
| CatBoost | 범주형 데이터 처리에 특화 |
| BPANN | 복잡한 비선형 관계 학습 가능 |


| Model | 논문 ROC AUC | 재현 ROC AUC |
|---|---|---|
| SVM | 0.977 | 0.974 |
| XGBoost | 0.984 | 0.986 |
| CatBoost | 0.985 | 0.990 |
| BPANN | 0.955 | 0.969 |

* 논문 재현 성공, CatBoost와 BPANN은 논문보다 성능 개선


결론
- CatBoost가 ROC AUC 0.990으로 가장 우수한 성능을 보임
- SVM은 SMOTE 적용으로 논문(Recall 0.886)보다 개선된 Recall(0.941)을 보임
  → 클래스 불균형 해소가 SVM 성능에 크게 기여함
- XGBoost, CatBoost는 논문과 유사하거나 더 높은 성능
- BPANN은 max_iter 조정으로 논문 대비 성능 개선

In [5]:
import joblib

joblib.dump(svm, '../data/svm_model.pkl')
joblib.dump(xgb, '../data/xgb_model.pkl')
joblib.dump(cat, '../data/cat_model.pkl')
joblib.dump(bpann, '../data/bpann_model.pkl')
print('모델 저장 완료!')

모델 저장 완료!
